Test integrating Pydantic with LlamaIndex. Use after inserting all the data with llamaindex_redis.ipynb

Load environment variables from .env file

In [10]:
import os

import nest_asyncio
from dotenv import load_dotenv

load_dotenv("../.env")
nest_asyncio.apply() # for async issues in Jupyter Notebook

Setup the embedding model

In [3]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [20]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=748206;https://logfire-us.pydantic.dev/iellis02/blue-horizon\https://logfire-us.pydantic.dev/iellis02/blue-horizon]8;;\

Connect to Redis Cloud

In [4]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.redis import RedisVectorStore
from redisvl.schema import IndexSchema

redis_conn_string = os.getenv("REDIS_URL")
schema = IndexSchema.from_dict(
    {
        "index": {"name": "blue_horizon", "prefix": "blue_horizon"},
        # customize fields that are indexed
        "fields": [
            # required fields for llamaindex
            {"type": "tag", "name": "id"},
            {"type": "tag", "name": "doc_id"},
            {"type": "text", "name": "text"},
            # custom vector field for bge-small-en-v1.5 embeddings
            {
                "type": "vector",
                "name": "vector",
                "attrs": {
                    "dims": 384,
                    "algorithm": "hnsw",
                    "distance_metric": "cosine",
                },
            },
        ],
    },
)
vector_store = RedisVectorStore(schema=schema, redis_url=redis_conn_string, overwrite=False)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

13:47:52 redisvl.index.index INFO   Index already exists, not overwriting.


In [45]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)
retriever = index.as_retriever(similarity_top_k=4)

In [46]:
retriever.retrieve("What swimming options are there?")

[NodeWithScore(node=TextNode(id_='FAQ000009', embedding=None, metadata={'category': 'amenities', 'subcategory': 'business', 'keywords': 'pool, swimming, recreation', 'last_updated': '2024-10-15'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs there a swimming pool?\n\nAnswer:\nYes, we have both indoor and outdoor pools open from 6:00 AM to 10:00 PM.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.6770186424260001),
 NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you off

In [53]:
from pydantic_ai import Agent

system_prompt = """You are an assistant who helps people find out information about a hotel.
Your sole job is to query the database for information about the hotel using the tool provided.
Provide the information returned from the tool that is relevant to the user's query
in a concise and well-formatted manner.

When you list options available for a service or amenity, be sure to mention that you
are only listing some of the options.

Do not offer to do anything except search for information about the hotel, but not on
the same topic.

Do not provide any instructions to the user concerning the hotel that were not provided
to you.
"""

agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=system_prompt,
)

@agent.tool_plain
def query_hotel_info(query: str):
    """Provide information about the hotel in response to a passed-in query.
    The query should be concise and not ask for many details.
    """
    retrieved_nodes = retriever.retrieve(query)

    return [{"metadata": node.metadata, "text": node.text} for node in retrieved_nodes]

In [54]:
result = await agent.run("What dining options are there?")
print(result.output)

17:10:10.135 agent run
17:10:10.137   chat gpt-5-mini
17:10:12.500   running 1 tool
17:10:12.501     running tool: query_hotel_info
17:10:12.561   chat gpt-5-mini
Here are some of the dining options and related details:

General
- Room service: Available 24/7 with a full menu during restaurant hours and a limited menu overnight.
- Breakfast: Yes — breakfast is included with most room rates. Please check your booking details.

Some specific dining offerings (only listing some options)
- Evening Dining
  - Price: 65
  - Duration/Service time: 45 minutes
  - Availability: 08:00–20:00
  - Location: Main Building
  - Booking required: No
  - Description: 45 minutes of professional service presentation to satisfy gourmet cravings.

- Dietary Specialist Menu
  - Price: 75
  - Duration/Service time: 45 minutes
  - Availability: 06:00–22:00
  - Location: Main Building
  - Booking required: Yes
  - Description: 45-minute culinary experience with dietary-specialist offerings and professional serv